In [ ]:
import numpy as np

from easydynamics.sample import Gaussian
from easydynamics.sample import Lorentzian
from easydynamics.sample import DampedHarmonicOscillator
from easydynamics.sample import Polynomial
from easydynamics.sample import SampleModel

import matplotlib.pyplot as plt

import scipp as sc


In [ ]:
# Creating a component
gaussian=Gaussian(name='Gaussian',center=0,width=0.5,area=1)
gaussian

In [ ]:
# Creating a sample model with multiple components
Sample= SampleModel(name='MySampleModel')
Sample.add_component(Gaussian(name='Gaussian',center=0.5,width=0.5,area=1))
Sample.add_component(Lorentzian(name='Lorentzian',center=0, width=0.5, area=1))
Sample.add_component(Polynomial(name='BG',coefficients=[0.1, 0.1]))
Sample.add_component(DampedHarmonicOscillator(name='DHO',center=1, width=0.3, area=1.2))

x=np.linspace(-2, 2, 100)
y=Sample.evaluate(x)
plt.plot(x, y, label='Sum of components')
plt.plot(x, Sample.evaluate_component('BG',x), label='Background')
plt.plot(x, Sample.evaluate_component('Gaussian',x), label='Gaussian')
plt.plot(x, Sample.evaluate_component('Lorentzian',x), label='Lorentzian')
plt.plot(x, Sample.evaluate_component('DHO',x), label='DHO')
plt.legend()
plt.xlabel('x')
plt.ylabel('y')


In [ ]:
# The area under the DHO curve is indeed equal to the area parameter.
xx=np.linspace(-10, 10, 10000)
yy=Sample.evaluate_component('DHO',xx)
area= np.trapezoid(yy, xx)
print(f"Area under DHO curve: {area:.4f}")


In [ ]:
# Creating a sample model with multiple components with different units
Sample= SampleModel(name='MySampleModel')
Sample.add_component(Gaussian(name='Gaussian',center=0.5*1e-3,width=0.5*1e-3,area=1*1e-3,unit='eV'))
Sample.add_component(Lorentzian(name='Lorentzian',center=0, width=500, area=1000,unit='microeV'))
Sample.add_component(Polynomial(name='BG',coefficients=[0.1, 0.1],unit='meV'))
Sample.add_component(DampedHarmonicOscillator(name='DHO',center=1, width=0.3, area=1.2,unit='meV'))

# y = sc.linspace('y', 0.0, 1.0, num=4, unit='m')
x=sc.linspace('x',-2, 2, num=100,unit='meV')
y=Sample.evaluate(x)
plt.plot(x, y, label='Sum of components')
plt.plot(x, Sample.evaluate_component('BG',x), label='Background')
plt.plot(x, Sample.evaluate_component('Gaussian',x), label='Gaussian')
plt.plot(x, Sample.evaluate_component('Lorentzian',x), label='Lorentzian')
plt.plot(x, Sample.evaluate_component('DHO',x), label='DHO')
plt.legend()
plt.xlabel('x (meV)')
plt.ylabel('y (arb. unit)')


In [ ]:
from easyscience import Parameter
import numpy as np

import scipp as sc

temperature=0.0001
temperature_unit='K'
omega=np.linspace(-1,1,100)
omega=1.0
omega_unit='meV'


kB=sc.scalar(value=8.617333262145e-2, unit='meV/K')  # Boltzmann constant in meV/K


if not isinstance(omega, sc.Variable):
    if isinstance(omega, (int, float)):  # Check if omega is an int or float
        omega = sc.scalar(value=float(omega), unit=omega_unit)
    elif isinstance(omega, (list, tuple, np.ndarray)):  # Check if omega is iterable
        if len(omega) == 1:
            omega = sc.scalar(value=omega[0], unit=omega_unit)
        else:
            omega = sc.array(dims=['x'], values=omega, unit=omega_unit)

if not isinstance(temperature,sc.Variable):
    if isinstance(temperature,Parameter):
        temperature=temperature.value
        temperature_unit=temperature.unit
    temperature=sc.scalar(value=temperature, unit=temperature_unit)

x = omega / (kB * temperature)

# Use masks for different regimes
DBF = sc.zeros_like(omega)

# Small omega: Taylor expansion
small = sc.abs(x) < 0.01

DBF = sc.where(small, kB * temperature + omega / 2 + omega**2 / (12 * kB * temperature), DBF)

# Large omega: asymptotic form
large = x > 50
DBF = sc.where(large, omega, DBF)

# General case: exact formula
mid = ~small & ~large
DBF = sc.where(mid, omega / (1 - sc.exp(-x)), DBF)

# Normalize by kB*T to get dimensionless - also makes the value 1 at omega=0
DBF=DBF/(kB*temperature)



In [ ]:
%matplotlib widget
from easyscience import Parameter
import numpy as np

import scipp as sc

from easydynamics.utils import detailed_balance_factor

temperatures=[0.1, 1, 10, 100]
temperature_unit='K'
omega=np.linspace(-1,1,100)
# omega=1.0
omega_unit='meV'

plt.figure()
for temperature in temperatures:
    DBF = detailed_balance_factor(omega, temperature, omega_unit,temperature_unit)
    plt.plot(omega, DBF.values, label=f'T={temperature}K')
plt.legend()
plt.xlabel('Energy transfer (meV)')
